# ¿Se pueden sumar las distorsiones de los puentes Einstein-Rosen?

**Proyecto:** *Vacuum Particle Journey*, línea A (mecanismo), continuación de *quantum-vacuum-journey* (notebooks 03b y 06).

**La idea.** Si cada par entrelazado está unido por un puente ER (ER = EPR), y cada puente distorsiona un poco el espacio-tiempo a su alrededor, ¿qué ocurre al sumar las distorsiones de muchísimos pares?

**Por qué se pueden sumar.** Lejos de los agujeros negros, la gravedad es débil: la métrica es $g_{\mu\nu} = \eta_{\mu\nu} + h_{\mu\nu}$ con $|h| \ll 1$. En una galaxia, $|h| \sim \Phi/c^2 \sim 10^{-6}$. En ese régimen las ecuaciones de Einstein son **lineales** y las distorsiones de muchas fuentes se suman:
$$h_{\mu\nu}^{total} = \sum_i h_{\mu\nu}^{(i)} \qquad\Longrightarrow\qquad \Phi_{total}(\mathbf{x}) = \sum_i \Phi_i(\mathbf{x})$$

**Lo que hace este notebook:**
1. Comprueba que el régimen lineal se cumple en una galaxia.
2. Muestra **qué se cancela y qué se acumula** al sumar millones de puentes orientados al azar.
3. Suma los puentes en tres escenarios:
   - **A.** Los pares siguen a la materia.
   - **B.** Los pares llenan el vacío de forma uniforme (aquí importa la presión, no solo la energía).
   - **C.** Distorsiones con signo aleatorio.
4. Calcula **dónde tendrían que estar los puentes** para explicar las curvas de rotación, y lo compara con los escenarios.
5. Lo muestra en 3D.

Modelo de cada puente: dos bocas de masa positiva (como en un puente Einstein-Rosen clásico), separadas una distancia con la distribución del núcleo de los notebooks anteriores (λ = 0.7 kpc). Unidades: kpc, km/s, masas solares.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.special import i0, i1, k0, k1
try:
    import plotly.graph_objects as go
    HAVE_PLOTLY = True
except ImportError:
    HAVE_PLOTLY = False
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "axes.edgecolor": "#c3c2b7", "lines.linewidth": 2,
                     "font.size": 10, "legend.frameon": False})
C_B, C_A, C_RAR, C_UNI, C_RND, C_GREY = "#8a8a85", "#eda100", "#1baf7a", "#2a78d6", "#e34948", "#9a9a95"
G = 4.30091e-6; C_KMS = 299792.458; TO_MS2 = 1e6 / 3.0857e19
RNG = np.random.default_rng(7)

# Galaxia de referencia: disco exponencial
MD, RD = 5e10, 3.0
Z0 = 0.196 * RD ** 0.633
LAM = 0.7                  # longitud típica de un puente (núcleo de los notebooks anteriores)
G_DAG = 1.0e-10 / TO_MS2   # escala de la RAR ajustada a SPARC
def v2_disk(r):
    y = r / (2 * RD)
    return 4 * np.pi * G * (MD / (2 * np.pi * RD**2)) * RD * y**2 * (i0(y) * k0(y) - i1(y) * k1(y))
def rar(gb):
    return gb / (-np.expm1(-np.sqrt(gb / G_DAG)))
R = np.linspace(0.3, 40, 160)
print(f"Disco: M = {MD:.1e} M☉, R_d = {RD} kpc, grosor z0 = {Z0:.2f} kpc")

## 1. ¿Es lineal la gravedad en una galaxia?

El parámetro que mide lo fuerte que es la gravedad es $|\Phi|/c^2$. Si es mucho menor que 1, las distorsiones se suman sin error apreciable; las correcciones no lineales son del orden de $(\Phi/c^2)^2$.

In [ ]:
v_max = np.sqrt(v2_disk(R)).max()
phi_c2 = (2.5 * v_max) ** 2 / C_KMS**2       # |Φ| en el centro ≈ unas pocas v²
print(f"|Φ|/c² en el centro de la galaxia ≈ {phi_c2:.1e}  → corrección no lineal ≈ {phi_c2**2:.0e}")
print("Conclusión: en una galaxia, sumar distorsiones es exacto a mejor que una parte en un millón.")

## 2. Qué se cancela y qué se acumula

Cada distorsión se puede descomponer en partes con distinta "forma":
- **Monopolo** (la parte que actúa como masa): siempre del mismo signo para masas positivas. Al sumar $N$ puentes crece como $N$.
- **Dipolos y cuadrupolos** (las partes con dirección): con orientaciones al azar se cancelan. Su suma crece solo como $\sqrt N$, y además decae más rápido con la distancia ($1/r^3$ frente a $1/r$).
- **Partes con signo aleatorio** (si algunas distorsiones fueran "negativas"): también crecen como $\sqrt N$.

Lo comprobamos sumando el campo de $N$ puentes en un punto lejano.

In [ ]:
def field_at(p, pos, m, soft=0.05):
    d = p[None, :] - pos
    r2 = (d**2).sum(1) + soft**2
    return -(G * m[:, None] * d / r2[:, None] ** 1.5).sum(0)

def bridges(n, centers_fn, lam=LAM, m_each=1.0, signs=None):
    c = centers_fn(n)
    u = RNG.normal(size=(n, 3)); u /= np.linalg.norm(u, axis=1)[:, None]
    L = RNG.gamma(2.0, lam, n)
    a, b = c + 0.5 * L[:, None] * u, c - 0.5 * L[:, None] * u
    m = np.full(n, m_each) if signs is None else m_each * signs
    return np.r_[a, b], np.r_[m, m]

ball = lambda n: RNG.normal(scale=1.0, size=(n, 3))
probe = np.array([20.0, 0, 0])
rows = []
for n in [100, 1000, 10000, 100000]:
    reps = 8 if n < 100000 else 4
    shape_, sign_, tot_, mono_ = [], [], [], []
    for _ in range(reps):
        pos, m = bridges(n, ball)
        g_all = field_at(probe, pos, m)
        g_mono = field_at(probe, np.zeros((1, 3)), np.array([m.sum()]))
        pos_s, m_s = bridges(n, ball, signs=RNG.choice([-1.0, 1.0], n))
        tot_.append(np.linalg.norm(g_all)); mono_.append(np.linalg.norm(g_mono))
        shape_.append(np.linalg.norm(g_all - g_mono)); sign_.append(np.linalg.norm(field_at(probe, pos_s, m_s)))
    rows.append(dict(N=n, total=np.mean(tot_), solo_monopolo=np.mean(mono_),
                     resto_con_forma=np.sqrt(np.mean(np.square(shape_))), signo_aleatorio=np.sqrt(np.mean(np.square(sign_)))))
T2 = pd.DataFrame(rows)
print(T2.to_string(index=False, float_format=lambda x: f"{x:.3e}"))
sl = lambda col: np.polyfit(np.log(T2.N), np.log(T2[col]), 1)[0]
print(f"\n(promedio de varias realizaciones) Crecimiento con N:  monopolo ∝ N^{sl('solo_monopolo'):.2f} · forma (dipolo/cuadrupolo) ∝ N^{sl('resto_con_forma'):.2f} · "
      f"signo aleatorio ∝ N^{sl('signo_aleatorio'):.2f}")
print("→ Al sumar muchos puentes solo sobrevive la parte que actúa como masa: la suma es, en la práctica, 'cuánta energía hay y dónde está'.")

## 3. Escenario A: los pares siguen a la materia

Repartimos $N$ puentes según la masa del disco, con una energía total igual a 5.4 veces la masa visible (la proporción cósmica Ω_c/Ω_b). Sumamos su campo en el plano del disco y lo comparamos con la curva plana que piden los datos (la RAR).

In [ ]:
def disk_points(n):
    Rr = RNG.gamma(2.0, RD, n); ph = RNG.uniform(0, 2 * np.pi, n); z = RNG.laplace(0, Z0, n)
    return np.c_[Rr * np.cos(ph), Rr * np.sin(ph), z]

def radial_profile(pos, m, radii, n_az=8):
    out = []
    for r in radii:
        g = 0.0
        for ph in np.linspace(0, 2 * np.pi, n_az, endpoint=False):
            p = np.array([r * np.cos(ph), r * np.sin(ph), 0.0])
            g += -field_at(p, pos, m, soft=0.15) @ (p / r)
        out.append(g / n_az)
    return np.array(out)

KAPPA, NB = 5.4, 40000
posA, mA = bridges(NB, disk_points, m_each=KAPPA * MD / (2 * NB))
Rs = R[::4]
gA = radial_profile(posA, mA, Rs)
vb2 = v2_disk(R)
v_A = np.sqrt(np.clip(np.interp(R, Rs, Rs * gA) + vb2, 0, None))
v_rar = np.sqrt(R * rar(vb2 / R))

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(R, np.sqrt(vb2), "--", color=C_B, label="solo materia visible")
ax.plot(R, v_A, color=C_A, label="A: + puentes que siguen a la materia (κ = 5.4)")
ax.plot(R, v_rar, color=C_RAR, label="curva plana que piden los datos (RAR)")
ax.set_xlabel("radio [kpc]"); ax.set_ylabel("velocidad [km/s]")
ax.set_title("Si los puentes siguen a la materia, la curva sube demasiado y luego cae", loc="left"); ax.legend(fontsize=9)
plt.show()
out = R > 20
slope_A = np.polyfit(np.log(R[out]), np.log(v_A[out]), 1)[0]; slope_R = np.polyfit(np.log(R[out]), np.log(v_rar[out]), 1)[0]
print(f"Pico: escenario A {v_A.max():.0f} km/s frente a {v_rar.max():.0f} km/s de la RAR")
print(f"Pendiente de la curva entre 20 y 40 kpc: escenario A v ∝ r^{slope_A:.2f} (cae, como r^-0.5) · RAR v ∝ r^{slope_R:.2f} (plana)")
print("→ El problema no es la cantidad de energía sino la forma: los puentes concentrados donde está la materia no dan una curva plana.")

## 4. Escenario B: pares uniformes en el vacío — aquí la presión cambia el signo

Si los pares salen del vacío por todas partes por igual, su suma es **uniforme**. En relatividad general, lo que atrae no es solo la energía: es $\rho + 3p/c^2$ (energía más tres veces la presión).
- Pares que se comportan como materia ($p \approx 0$): **atraen**.
- Energía del vacío ($p = -\rho c^2$, la misma en todas direcciones y para cualquier observador): $\rho + 3p/c^2 = -2\rho$, y **repele**. Es la constante cosmológica Λ.

Una suma uniforme no forma halos alrededor de las galaxias: actúa sobre el universo entero. Por eso, en este escenario los puentes serían candidatos a **energía oscura**, no a materia oscura.

In [ ]:
KG_M3_PER_MSUN_KPC3 = 1.989e30 / (3.0857e19) ** 3
rho_L = 5.8e-27 / KG_M3_PER_MSUN_KPC3                  # densidad de energía oscura observada [M☉/kpc³]
rho_disk_mid = MD / (4 * np.pi * RD**2 * Z0)
v2_L = -(8 * np.pi * G / 3) * rho_L * R**2              # Λ repele: resta velocidad
print(f"Energía oscura observada: {rho_L:.0f} M☉/kpc³  (el centro del disco: {rho_disk_mid:.1e} M☉/kpc³, {rho_disk_mid/rho_L:.0e} veces más)")
print(f"Efecto en la galaxia a 35 kpc: cambia v² en {abs(np.interp(35, R, v2_L)):.1f} (km/s)², frente a {v_rar[-1]**2:.0f} (km/s)² → despreciable")

# El problema de la constante cosmológica: la suma ingenua de pares del vacío
rhoL_GeV4 = (2.25e-12) ** 4
rows = [dict(corte=nm, escala_GeV=E, densidad_ingenua_GeV4=E**4, veces_la_observada=f"10^{np.log10(E**4/rhoL_GeV4):.0f}")
        for nm, E in [("Planck", 1.22e19), ("electrodébil", 246.0), ("QCD (fuerza fuerte)", 0.2), ("masa del electrón", 0.511e-3)]]
print("\nSuma ingenua de la energía de los pares del vacío hasta una energía de corte (orden de magnitud):")
print(pd.DataFrame(rows).to_string(index=False))
print("→ Sumar los pares del vacío da demasiado, incluso con el corte más bajo: es el problema de la constante cosmológica, sin resolver.")

## 5. Escenario C: distorsiones con signo aleatorio

Si algunos puentes distorsionaran el espacio "hacia fuera" y otros "hacia dentro" en igual proporción (lo que exigiría masa negativa en algunas bocas, algo que un puente Einstein-Rosen clásico no tiene), la suma en promedio sería cero y solo quedarían fluctuaciones de tamaño $1/\sqrt N$.

In [ ]:
posC, mC = bridges(NB, disk_points, m_each=KAPPA * MD / (2 * NB), signs=RNG.choice([-1.0, 1.0], NB))
gC = radial_profile(posC, mC, Rs)
print(f"Escenario C: velocidad extra cuadrática media = {np.sqrt(np.mean(np.abs(Rs * gC))):.1f} km/s "
      f"(frente a {np.sqrt(np.mean(Rs * gA)):.0f} km/s del escenario A con la misma energía total)")
print("→ Con signos al azar, la suma prácticamente se anula: no puede sostener una curva plana.")

## 6. ¿Dónde tendrían que estar los puentes?

Invertimos el problema: ¿qué densidad de puentes haría falta para producir exactamente la curva plana que piden los datos? A partir de la masa extra que exige la RAR,
$$G M_{extra}(r) = r^2\big[g_{RAR}(g_b) - g_b\big],\qquad \rho_{extra}(r) = \frac{1}{4\pi G r^2}\frac{d\,(G M_{extra})}{dr}$$
y la comparamos con la densidad de la materia visible. El cociente es **cuántos puentes por unidad de materia** se necesitarían en cada radio.

In [ ]:
gb = vb2 / R
GMx = R**2 * (rar(gb) - gb); GMb = R * vb2
rho_x = np.gradient(GMx, R) / (4 * np.pi * G * R**2)
x_ = R / RD
rho_b = MD * x_ * np.exp(-x_) / RD / (4 * np.pi * R**2)       # disco exponencial: dM/dr exacto, repartido en esferas
ratio = rho_x / rho_b
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.semilogy(R, rho_b, color=C_B, label="materia visible (equivalente esférico)")
a1.semilogy(R, np.clip(rho_x, 1e-3, None), color=C_RAR, label="puentes necesarios")
a1.semilogy(R, 5.4 * rho_b, "--", color=C_A, label="escenario A (5.4 × materia)")
a1.semilogy(R, 0 * R + rho_L, ":", color=C_UNI, label="energía oscura (uniforme)")
a1.set_xlabel("radio [kpc]"); a1.set_ylabel("densidad [M☉/kpc³]"); a1.set_title("Densidad necesaria frente a los escenarios", loc="left"); a1.legend(fontsize=8.5)
a2.semilogy(R, np.clip(ratio, 1e-3, None), color=C_RAR)
a2.axhline(5.4, ls="--", color=C_A, lw=1.2); a2.text(30, 6.5, "5.4 (proporción cósmica)", color=C_A, fontsize=9)
a2.set_xlabel("radio [kpc]"); a2.set_ylabel("puentes necesarios por unidad de materia")
a2.set_title("Cada vez más puentes donde hay menos materia", loc="left")
plt.tight_layout(); plt.show()
for rr in [RD, 3 * RD, 6 * RD, 10 * RD]:
    print(f"r = {rr:4.1f} kpc ({rr/RD:.0f} R_d): {np.interp(rr, R, ratio):10.3g} puentes por unidad de materia   g_b = {np.interp(rr, R, gb)*TO_MS2:.1e} m/s²")
slope = np.polyfit(np.log(R[R > 4 * RD]), np.log(rho_x[R > 4 * RD]), 1)[0]
print(f"Fuera del disco, la densidad necesaria cae como r^{slope:.2f} (un halo, ∝ r^-2), no como el disco (exponencial).")

## 7. Vista 3D: dónde están los puentes en cada caso

Izquierda conceptual de la hipótesis: **naranja** = puentes que siguen a la materia (escenario A); **verde** = puentes distribuidos como exigen los datos (sección 6). Los puntos grises son la materia del disco. Se puede girar y hacer zoom.

In [ ]:
n_show = 500
pa, _ = bridges(n_show, disk_points)
cum = np.cumsum(np.clip(rho_x, 0, None) * 4 * np.pi * R**2 * np.gradient(R)); cum /= cum[-1]
r_need = np.interp(RNG.uniform(0, 1, n_show), cum, R)
u = RNG.normal(size=(n_show, 3)); u /= np.linalg.norm(u, axis=1)[:, None]
pn, _ = bridges(n_show, lambda n: r_need[:, None] * u)
stars = disk_points(1500)
def seg(pos):
    a, b = pos[:n_show], pos[n_show:]
    xs, ys, zs = [], [], []
    for p, q in zip(a, b):
        xs += [p[0], q[0], None]; ys += [p[1], q[1], None]; zs += [p[2], q[2], None]
    return xs, ys, zs
if HAVE_PLOTLY:
    tr = [go.Scatter3d(x=stars[:, 0], y=stars[:, 1], z=stars[:, 2], mode="markers", marker=dict(size=1.5, color=C_B, opacity=0.5), name="materia del disco")]
    for pos, col, nm in [(pa, C_A, "A: puentes que siguen a la materia"), (pn, C_RAR, "puentes donde los pide la curva plana")]:
        xs, ys, zs = seg(pos); tr.append(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line=dict(color=col, width=3), name=nm))
    fig = go.Figure(tr)
    fig.update_layout(title="Dónde tendrían que estar los puentes", height=620, margin=dict(l=0, r=0, t=40, b=0),
                      scene=dict(aspectmode="data", xaxis_title="x [kpc]", yaxis_title="y [kpc]", zaxis_title="z [kpc]"))
    fig.show()
else:
    f = plt.figure(figsize=(7, 6)); ax = f.add_subplot(projection="3d")
    ax.scatter(*stars.T, s=1, c=C_B, alpha=0.4)
    for pos, col in [(pa, C_A), (pn, C_RAR)]:
        for p, q in zip(pos[:n_show], pos[n_show:]): ax.plot(*np.c_[p, q], color=col, lw=0.6)
    plt.show()
print(f"Radio mediano: puentes que siguen a la materia {np.median(np.linalg.norm(pa[:n_show], axis=1)):.1f} kpc · "
      f"puentes necesarios {np.median(r_need):.1f} kpc")

## 8. A dónde nos lleva el camino

| Pregunta | Respuesta del cálculo |
|---|---|
| ¿Se pueden sumar las distorsiones? | Sí: en una galaxia la gravedad es lineal a mejor que una parte en un millón |
| ¿Qué sobrevive a la suma? | Solo la parte que actúa como masa (∝ N). Las formas con dirección y los signos al azar se cancelan (∝ √N) |
| Puentes que siguen a la materia (A) | Demasiada fuerza dentro, demasiado poca fuera: no da curvas planas |
| Puentes uniformes del vacío (B) | Con presión negativa repelen: son **energía oscura**, no materia oscura. La suma ingenua es 10⁴⁰–10¹²⁰ veces demasiado grande |
| Signos aleatorios (C) | La suma se anula |
| ¿Dónde harían falta? | En un **halo** que cae como 1/r², con cada vez más puentes por unidad de materia lejos del disco, justo donde la gravedad es más débil |

**La lectura para la hipótesis.** Sumar distorsiones funciona, pero el resultado depende solo de **dónde se crean los pares y con qué ecuación de estado** (presión). Para que la red de puentes sea la materia oscura, haría falta un mecanismo que cree o conserve puentes **preferentemente donde la gravedad es débil**, siguiendo aproximadamente $g_b \lesssim 10^{-10}$ m/s². Es la misma condición que encontramos de forma empírica en el notebook de κ(g), ahora deducida desde la suma.

**Dos caminos que salen de aquí:**
1. **Materia oscura:** buscar un mecanismo físico por el que la creación o la supervivencia de pares entrelazados dependa de la aceleración local. Por ejemplo, el efecto Unruh asocia una temperatura a la aceleración, $T = \hbar a/(2\pi c k_B)$; a aceleraciones de ~10⁻¹⁰ m/s² esa temperatura es diminuta. ¿Hay una escala en la que la aceleración de fondo del universo ($\sim cH_0$) compita con la local? Es la intuición que también está detrás de MOND y de Verlinde.
2. **Energía oscura:** estudiar por qué la suma de los pares del vacío da un valor tan pequeño (o se cancela casi del todo). Ahí tu idea entraría en el problema de la constante cosmológica, uno de los grandes problemas abiertos de la física.